<a href="https://colab.research.google.com/github/scostavinicius/lean-agent/blob/tools/lean_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente provador de teoremas em Lean 4

**Dupla:** VINICIUS COSTA SOARES · MATHEUS VINÍCIUS SILVA FREIRE DE CASTRO

Tarefa da Unidade I — IA Agêntica (2026.2)

## Sobre este notebook

Por enquanto, este notebook só prepara o ambiente. As seções do agente estão vazias e serão preenchidas pela dupla ao longo da semana, uma de cada vez.

Cada célula de código começa com um comentário **Por quê**, explicando o motivo de ela existir. Ao adicionar uma célula nova, mantenham esse hábito: ele ajuda o parceiro a entender o que foi feito e já adianta o relatório.

## 1 — Ambiente

Três coisas precisam funcionar antes de escrever qualquer agente: a biblioteca da disciplina, o Lean e o acesso ao modelo. Rodem as células em ordem. Se todas terminarem sem erro, o ambiente está pronto.

In [ ]:
!pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

In [ ]:
!curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.33.0

import os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + os.pathsep + os.environ["PATH"]
!lean --version

In [ ]:
# confirmar que o Lean verifica uma prova de verdade, antes de
# colocar um agente no meio. Se aparecer só "código de retorno: 0", deu certo.
# Experimente trocar "omega" por "rfl" ou por "sorry" e rodar de novo.
from pathlib import Path
import subprocess

Path("Teste.lean").write_text("""
theorem teste (a b : Nat) : a + b = b + a := by
  omega
""")
resultado = subprocess.run(["lean", "Teste.lean"], capture_output=True, text=True)
print("código de retorno:", resultado.returncode)
print(resultado.stdout + resultado.stderr)

In [ ]:
# Usamos o gpt-oss-120bpelo plano gratuito do Groq.
# A tarefa proíbe a chave no notebook. Por isso ela é lida de um segredo do
# Colab chamado GROQ_API_KEY (ícone de chave na barra lateral) ou de uma
# variável de ambiente com esse nome.
from agentkit import LLMAPI

try:
    from google.colab import userdata
    os.environ.setdefault("GROQ_API_KEY", userdata.get("GROQ_API_KEY"))
except ImportError:
    pass  # fora do Colab, a variável de ambiente já deve existir

llm = LLMAPI(
    "openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0.0,
    max_tokens=2000,
)
print(llm.invoke("Responda apenas: ok"))

In [ ]:
import os
from google.colab import userdata


os.environ["GEMINI_API_KEY"] = ""

try:
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    gemini_key = os.environ.get("GEMINI_API_KEY", "")

llm = LLMAPI(
    "gemini-3.5-flash-lite",
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    temperature=0.0,
    max_tokens=2000,
)
print(llm.invoke("Responda apenas: ok"))

## 2 — O agente

Cada seção abaixo corresponde a uma parte do agente. A ordem sugerida é a das seções. Ao começar uma, escrevam na própria seção quem está trabalhando nela.

### 2.1 Casos de teste

**O que vai aqui:** Os teoremas que o agente vai tentar provar e o resultado esperado de cada um.

**Requisito atendido:** 10 casos de teste, pelo menos 3 difíceis.

### 2.2 Verificação no Lean

**O que vai aqui:** Uma função que recebe um enunciado e uma prova, roda o Lean e diz se a prova foi aceita.

**Requisito atendido:** é o que torna a tarefa verificável.

### 2.3 Ferramentas

**O que vai aqui:** Funções com `@tool` que o agente pode chamar.

**Requisito atendido:** pelo menos 3 ferramentas feitas pela dupla.

In [ ]:
from agentkit import tool

In [ ]:
@tool
def get_arquivo_lean(enunciado: str, prova: str, nome_arquivo: str) -> None:
    """Gera arquivo no formato lean com nome definido como {nome_arquivo}.lean."""
    prova_indentada = "\n".join("  " + linha for linha in prova.strip().splitlines())
    codigo = f"theorem alvo {enunciado} := by\n{prova_indentada}\n"
    Path(f"{nome_arquivo}.lean").write_text(codigo, encoding="utf-8")

@tool
def ler_arquivo_lean(nome_arquivo: str) -> str:
    """Lê arquivo no formato lean"""
    return Path(f"{nome_arquivo}.lean").read_text(encoding="utf-8")

In [ ]:
@tool
def test_prova(enunciado: str, prova: str, nome_arquivo: str = "Prova") -> tuple[bool, str]:
    """Roda o Lean em `theorem alvo <enunciado> := by <prova>` e devolve (aceita, saída do Lean)."""
    caminho = Path(nome_arquivo)
    if caminho.is_file():
        ler_arquivo_lean(str(caminho))

    get_arquivo_lean(enunciado, prova, nome_arquivo)
    arquivo = nome_arquivo + ".lean"
    resultado = subprocess.run(["lean", arquivo], capture_output=True, text=True)

    saida = resultado.stdout + resultado.stderr
    aceita = resultado.returncode == 0 and "sorry" not in saida and "sorry" not in prova

    return aceita, saida

In [ ]:
print(test_prova("(a b : Nat) : a + b = b + a", "omega"))   # esperado: True
print(test_prova("(a b : Nat) : a + b = b + a", "rfl"))            # esperado: False, com o erro
print(test_prova("(a b : Nat) : a + b = b + a", "sorry"))          # esperado: False

In [ ]:
@tool
def resolver_teorema(enunciado: str, prova: str, nome_arquivo: str):
  """Verifica um teorema em Lean 4 ou aciona o agente iterativo se houver 'sorry'."""
  if "sorry" in prova:
        return agente_sorry(enunciado, prova, nome_arquivo)
  else:
        return test_prova(enunciado, prova, nome_arquivo)


In [ ]:
@tool
def check(expression: str) -> str:
    """Roda o Lean em `#check <expression>` e devolve o Type do Lean.

    Em caso de string, use aspas simples seguido de aspas duplas
    Exemplo: '"string"'
    """
    codigo = f"#check {expression}\n"

    nome_arquivo = "Check.lean"
    Path(nome_arquivo).write_text(codigo, encoding="utf-8")

    arquivo = nome_arquivo
    resultado = subprocess.run(["lean", arquivo], capture_output=True, text=True)

    saida = resultado.stdout + resultado.stderr
    return saida

In [ ]:

@tool
def ler_guia_taticas() -> str:
    """Retorna o guia rápido de sintaxe e táticas básicas do Lean 4."""
    caminho = Path("GuiaLean4.md")
    if caminho.exists():
        return caminho.read_text(encoding="utf-8")
    return "Guia não encontrado."

In [ ]:
print(check("4 + 3"))
print(check("Nat.add"))
print(check("Lean"))

In [ ]:
import time
@tool
def verificar_prova(enunciado: str, prova: str, nome_arquivo: str, max_iteracoes: int):
    """Verifica se a prova está correta. Se falhar, usa o erro do Lean para refinar a prova iterativamente."""

    aceita, saida = test_prova(enunciado, prova, nome_arquivo)

    prova_atual = prova
    max_iteracoes = max_iteracoes
    iteracao = 0

    while not aceita and iteracao < max_iteracoes:
        iteracao += 1
        # time.sleep(20) # comentado pq mudei para o gemini 1.5 ja q o groq acabou os tokens
        prova_atual = agente_sorry(enunciado, prova_atual, saida)
        aceita, saida = test_prova(enunciado, prova_atual, nome_arquivo)
        print(f"tentativa {iteracao}")
        print(prova_atual)
    return aceita, saida

In [ ]:
prova_atual = resolver_teorema("(a b c : Nat) : (a + b) + c = a + (b + c)", "sorry", "Prova.lean")
print(verificar_prova("(a b c : Nat) : (a + b) + c = a + (b + c)", prova_atual, "Prova", 10))

### 2.4 Prompt e contexto

**O que vai aqui:** As instruções que dizem ao modelo como trabalhar.

**Requisito atendido:** prompt e contexto projetados pela dupla.

In [ ]:
def agente_sorry(enunciado, prova_atual, erro_lean):
  """Usa o LLM para gerar uma nova tentativa de prova com base no erro do Lean, seguindo o guia estrito."""
  prompt = f"""
    Você é um assistente especialista em Lean 4.
    Estamos tentando provar o seguinte teorema:
    Teorema: {enunciado}

    =========================================
    DIRETRIZES DE TÁTICAS (GuiaLean4.md):
    =========================================
    1. Tática `intro`: Usada para introduzir variáveis ou hipóteses no contexto local (implicações/quantificadores). Ex: `intro hp hq`.
    2. Tática `rw` (rewrite): Substitui termos usando igualdades. Use colchetes e `←` para sentido inverso. Ex: `rw [Nat.add_zero]`, `rw [← lema]`.
    3. Tática `induction`: Usada para indução em tipos como Nat. Sintaxe:
       induction n with
       | zero => ...
       | succ n ih => ...
    4. Tática `omega`: Provador automático de aritmética linear para Nat/Int (resolve equações e somas lineares).
    5. Tática `simp`: Simplifica o alvo aplicando regras padrão e lemas marcados. Ex: `simp [Nat.add_comm]`.
    6. Tática `rfl` (reflexivity): Fecha metas identicamente iguais por definição.

    =========================================
    RESTRIÇÕES IMPORTANTES:
    =========================================
    - Não utilize teoremas externos complexos prontos que você não tenha provado. Foque em passos fundamentais usando as táticas acima.
    - Siga estritamente uma resolução linha por linha (passo a passo detalhado).

    =========================================
    ESTADO ATUAL:
    =========================================
    A tentativa atual de prova é:
    ```lean
    {prova_atual}
    ```

    O Lean retornou a seguinte saída/erro ao rodar essa prova:
    ```
    {erro_lean}
    ```

    =========================================
    INSTRUÇÕES DE SAÍDA:
    =========================================
    - Pense e gere um plano mental de passos antes de escrever o código.
    - Sua escrita durante as demonstrações e correções deve ser APENAS em código Lean 4 puro e detalhado.
    - Retorne APENAS o código das táticas que ficam após o "by". Não repita a palavra "theorem", o cabeçalho e nem a palavra "by" (as tools já fazem essa inserção).
    """

  response = llm.invoke(prompt)
  texto_limpo = response.replace("```lean", "").replace("```", "").strip()
  
  if texto_limpo.startswith("by"):
      texto_limpo = texto_limpo[2:].strip()
      
  return str(texto_limpo)

### 2.5 Mecanismos

**O que vai aqui:** Por exemplo, estado para não repetir tentativas e planejamento da prova em etapas.

**Requisito atendido:** pelo menos 2 mecanismos com função real.

In [ ]:
# Por quê:

### 2.6 Agente

**O que vai aqui:** Juntar modelo, ferramentas e prompt no `Agent` do agentkit.

**Requisito atendido:** o modelo decide quais ferramentas usar e quando parar.

In [ ]:
# Por quê:

### 2.7 Testes e resultados

**O que vai aqui:** Rodar os 10 casos e mostrar entrada, esperado, obtido, aprovado e a taxa de acerto.

**Requisito atendido:** tabela de testes e análise dos erros.

In [ ]:
# Por quê:

## 3 — Relatório (a escrever no fim)

### O que o agente faz

### Por que a tarefa é verificável

### Principais decisões de projeto

### Análise dos erros